<a href="https://colab.research.google.com/github/Dnyamwamu/neural_nets/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [5]:
def graph_attention(X, omega, beta, phi, A):
    N = X.shape[1]  # number of nodes

    # Step 1: Compute X' = ReLU(omega * X + beta)
    X_prime = ReLU(np.matmul(omega, X) + beta)

    # Step 2: Compute pairwise scores S
    # We'll compute [X'_i || X'_j]^T phi for all i,j
    S = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            concat = np.concatenate([X_prime[:, i], X_prime[:, j]])[:, np.newaxis]  # shape (2D,1)
            S[i,j] = np.matmul(phi.T, concat)[0,0]

    # Step 3: Mask S where there is no edge (including self loop)
    mask = A + np.eye(N)
    S_masked = np.where(mask > 0, S, -1e20)

    # Step 4: Apply softmax over columns to get attention weights
    attention = softmax_cols(S_masked)

    # Step 5: Multiply X' by attention values
    X_attended = np.matmul(X_prime, attention)

    # Step 6: Apply ReLU
    output = ReLU(X_attended)

    return output

In [6]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[0.192 0.208 0.422 0.258 0.572 0.286 0.065 0.736]
 [0.035 0.322 0.716 0.028 1.002 0.312 0.282 0.804]
 [1.206 0.906 0.311 1.127 0.347 0.287 0.059 0.739]
 [1.208 0.787 0.    0.978 0.042 0.037 0.058 0.096]]


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.


In [7]:
def graph_dot_product_attention(X, W_Q, W_K, W_V, A):
    """
    X : (D, N) node feature matrix
    W_Q, W_K, W_V : (D,D) weight matrices
    A : (N,N) adjacency matrix (0/1)
    Returns updated node features of shape (D, N)
    """
    N = X.shape[1]  # number of nodes

    # Step 1: Compute Queries, Keys, and Values
    Q = np.matmul(W_Q, X)  # shape (D, N)
    K = np.matmul(W_K, X)  # shape (D, N)
    V = np.matmul(W_V, X)  # shape (D, N)

    # Step 2: Compute raw attention scores (dot-product)
    S = np.matmul(Q.T, K) / np.sqrt(X.shape[0])  # shape (N, N)

    # Step 3: Mask scores for non-neighbors (including self)
    mask = A + np.eye(N)
    S_masked = np.where(mask > 0, S, -1e20)

    # Step 4: Apply softmax column-wise
    attention = softmax_cols(S_masked)

    # Step 5: Aggregate values
    X_attended = np.matmul(V, attention)

    # Step 6: Apply ReLU activation
    output = ReLU(X_attended)

    return output